In [7]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
if '_id' in df.columns:
    df.drop(columns=['_id'],inplace=True)
    

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)
def get_rescue_data(rescue_type):
    #Defaault all animals
    query = {}
    
    if rescue_type == 'water':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "Labrador Retriever Mix",
                "Chesapeake Bay Retriever",
                "Newfoundland"
            ]},
             "age_upon_outcome_in_weeks": {"$lte": 156}
        }
    elif rescue_type == 'mountain':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "German Shepard",
                "Alaskan Malamute",
                "Siberian Husky"
            ]},
            "age_upon_outcome_in_weeks": {"$lte": 156}
        }
    elif rescue_type == 'disaster':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "Doberman Pinscher",
                "German Shepard",
                "Golden Retriever"
            ]},
            "age_upon_outcome_in_weeks": {"$lte": 156}
        }
    dff = pd.DataFrame.from_records(db.read(query))
    
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
    return dff
#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('CS-340 Dashboard - Tammy Wilson'))),
    html.Hr(),
#Filter control
    dcc.Dropdown(
        id='filter-type',
        options=[
            {'label': 'Reset (All Animals)', 'value': 'reset'},
            {'label': 'Water Rescue', 'value': 'water'},
            {'label': 'Mountain/Wilderness Rescue', 'value': 'mountain'},
            {'label': 'Disaster/Individual Tracking', 'value': 'disaster'}
        ],
        value='reset',
        clearable=False,
        style={'width': '50%'}
    ),
    html.Hr(),
    
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable=False,
        row_selectable='single',
        row_deletable=False,
        selected_columns=[],
        selected_rows=[],
        page_action="native",
        page_current=0,
        page_size=10
    ),
#FIXME: Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here 


    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',style={'display' : 'flex'},children=[
        html.Div(id='graph-id',className='col s12 m6',),
        html.Div(id='map-id',className='col s12 m6',)
    ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(
    Output('datatable-id','data'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):
    if filter_type == 'reset':
        dff = pd.DataFrame.from_records(db.read({}))
    else:
        dff = get_rescue_data(filter_type)
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
    if dff.empty:
        return []
        
    return dff.to_dict('records')

@app.callback(
    Output('graph-id', 'children'),
    Input('datatable-id', "derived_virtual_data")
)
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return [html.Div("No data available.")]
    
    dff = pd.DataFrame(viewData)
    
#Simple bar chart: count by breed
    breed_col = None
    for col in dff.columns:
        if col.lower() == "breed":
            breed_col = col
            break
    if breed_col is None:
        return [html.Div("Breed column not found in data.")]
    
    if dff.empty:
        return [html.Div("No data available for this filter.")]
    
    breed_counts = dff[breed_col].value_counts().reset_index()
    breed_counts.columns = ['Breed', 'Count']
    
    fig = px.bar(breed_counts, x='Breed', y='Count',
                 title='Animals by Breed')
    
    return [dcc.Graph(figure=fig)]

@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
   
    #No data at all
    if viewData is None or len(viewData) == 0:
        return [html.Div("No map data available.")]
    dff = pd.DataFrame.from_dict(viewData)
    
    #Make sure enough columns for iloc[13] and [14]
    if dff.shape[1] <= 14:
        return [html.Div("Dataset does not contain columns 13 and 14 for map coordinates.")]
    #Validate selected row
    if index is None or len(index) ==0:
        row = 0
    else:
        row = index[0]
                         
    #validate row index
    if row >= len(dff):
        return [html.Div("Selected row is out of range.")]
                         
    #Extract coordinates
    lat = dff.iloc[row, 13]
    lon = dff.iloc[row, 14]
                         
    #Validate coordinate values
    if lat in [None, "", ""] or lon in [None, "", ""]:
        return[
    
    # Austin TX is at [30.75,-97.48]
            dl.Map(
                style={'width': '1000px', 'height': '500px'}, 
                center=[30.75, -97.48],
                zoom=10, 
                children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                   position=[30.75, -97.48],
                   children=[
                       dl.tooltip("No coordinates available"),
                       dl.Popup([
                           html.H1("Animal Name"),
                           html.P("No coordinate data for this entry.")
                
                        ])
                    ]
                )
            ]
        )
    ]
    #If coordinates are valid load real map
    return [
        dl.map(
            style={'width': '1000px', 'height': '500px'},
            center=[lat, lon],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.marker(
                    position=[lat, lon],
                    children=[
                        dl.ToolTip(dff.iloc[row, 4]),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row, 9])
                        ])
                    ]
                )
            ]
         )
      ]
if __name__ == '__main__':
# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
    app.run_server(debug=True) 

Dash app running on https://diegonice-invitenikita-3000.codio.io/proxy/8050/
